# 05-01 LangChain 基础

**LangChain** 是 Agent 开发的核心框架，提供 Prompt 模板、Chain 编排、工具调用等基础设施。

**本节目标**：
- 掌握 ChatPromptTemplate 和 LCEL（LangChain Expression Language）
- 实现 Prompt → LLM → OutputParser 的标准链
- 理解 Runnable 接口（invoke/stream/batch）

---

In [ ]:
import os, sys
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")

try:
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
    from langchain_core.runnables import RunnablePassthrough, RunnableLambda
    from langchain_openai import ChatOpenAI
    print("LangChain 导入成功")
    HAS_LANGCHAIN = True
except ImportError as e:
    print(f"LangChain 未安装: {e}")
    print("安装: pip install langchain langchain-openai langchain-anthropic")
    HAS_LANGCHAIN = False

## 1. ChatPromptTemplate

In [ ]:
if HAS_LANGCHAIN:
    # 定义广告审核 prompt 模板
    review_prompt = ChatPromptTemplate.from_messages([
        ("system", """你是B站广告审核专家。
审核标准：
1. 不得含极限词（最、第一、绝对）
2. 不得虚假宣传
3. 标题不超过30字
输出JSON格式：{{"approved": bool, "issues": [str], "score": float}}"""),
        ("user", "请审核以下广告内容：\n{ad_content}")
    ])
    
    # 格式化 prompt
    messages = review_prompt.format_messages(
        ad_content="B站独家最强游戏皮肤，限时5折，数量有限！"
    )
    for msg in messages:
        print(f"[{msg.type}]: {msg.content[:80]}...")
else:
    print("""
ChatPromptTemplate 用法：
  prompt = ChatPromptTemplate.from_messages([
      ("system", "你是...{role}"),   # 模板变量用 {}
      ("user", "{input}"),
  ])
  messages = prompt.format_messages(role="审核员", input="审核这个广告")
    """)

## 2. LCEL Chain（核心！）

LCEL 用 `|` 操作符连接组件，形成数据流 pipeline：
```python
chain = prompt | llm | output_parser
result = chain.invoke({"input": "..."})
```

In [ ]:
if HAS_LANGCHAIN and os.environ.get("OPENAI_API_KEY"):
    # 创建 LLM
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    
    # LCEL Chain: prompt → LLM → 字符串输出
    chain = review_prompt | llm | StrOutputParser()
    
    # 调用
    result = chain.invoke({"ad_content": "B站会员限时优惠，享受高清无广告体验"})
    print(f"审核结果: {result}")
else:
    print("""
LCEL Chain 核心概念：

  chain = prompt | llm | StrOutputParser()
  
  数据流：
  {"ad_content": "..."}
    → prompt.format_messages()   → [SystemMessage, HumanMessage]
    → llm.invoke()              → AIMessage(content="...")
    → StrOutputParser()         → "纯文本字符串"

三种调用方式：
  chain.invoke(input)         # 同步，单条
  chain.stream(input)        # 流式，逐 token
  chain.batch([in1, in2])    # 批量，并发
  await chain.ainvoke(input) # 异步
    """)

## 3. 结构化输出（Pydantic + JsonOutputParser）

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class AdReviewResult(BaseModel):
    """广告审核结果"""
    approved: bool = Field(description="是否通过审核")
    score: float = Field(ge=0, le=10, description="质量评分0-10")
    issues: list[str] = Field(default_factory=list, description="发现的问题")
    category: Literal["合规", "违规", "待修改"] = Field(description="审核分类")

if HAS_LANGCHAIN:
    parser = JsonOutputParser(pydantic_object=AdReviewResult)
    
    structured_prompt = ChatPromptTemplate.from_messages([
        ("system", "你是广告审核员。{format_instructions}"),
        ("user", "审核: {ad_content}")
    ])
    
    # 注入格式指令
    prompt_with_format = structured_prompt.partial(
        format_instructions=parser.get_format_instructions()
    )
    
    print("格式指令（发给LLM的）:")
    print(parser.get_format_instructions()[:200] + "...")
else:
    print("""
结构化输出模式：
  1. 定义 Pydantic 模型 → 自动生成 JSON Schema
  2. JsonOutputParser 注入格式指令到 prompt
  3. LLM 输出 JSON → Parser 自动验证并解析
  4. 返回 Python dict（或 Pydantic 对象）
  
  也可以用 llm.with_structured_output(AdReviewResult) 更简洁
    """)

## 4. RunnablePassthrough 与多步链

In [ ]:
if HAS_LANGCHAIN:
    from langchain_core.runnables import RunnableParallel
    
    # 多步链: 生成广告 → 审核 → 评估
    generate_prompt = ChatPromptTemplate.from_messages([
        ("system", "你是B站广告文案师，用15字以内写一个广告标题"),
        ("user", "为以下产品写广告标题: {product}")
    ])
    
    # 链式组合
    generate_chain = generate_prompt | llm | StrOutputParser()
    
    # RunnablePassthrough 保留原始输入
    full_chain = (
        RunnablePassthrough.assign(title=generate_chain)  # 生成标题，保留原始 product
        | RunnableLambda(lambda x: print(f"  生成标题: {x['title']}") or x)
        | review_prompt.partial()  # 送去审核
    )
    
    print("多步链已构建（generate → review）")
else:
    print("""
多步链组合:
  # 保留中间结果
  chain = (
      RunnablePassthrough.assign(step1_result=chain1)
      | RunnableLambda(lambda x: process(x))
      | chain2
  )
  
  # 并行执行
  parallel = RunnableParallel(
      analysis=analysis_chain,
      summary=summary_chain,
  )
  result = parallel.invoke({"input": "..."})
    """)

## LCEL 核心操作

| 操作 | 说明 | 示例 |
|------|------|------|
| `a \| b` | 顺序连接 | `prompt \| llm \| parser` |
| `.invoke(input)` | 同步执行 | `chain.invoke({"q": "..."})`|
| `.stream(input)` | 流式输出 | `for chunk in chain.stream(...)` |
| `.batch(inputs)` | 批量并发 | `chain.batch([in1, in2])` |
| `RunnablePassthrough` | 透传输入 | 保留原始数据同时添加中间结果 |
| `RunnableParallel` | 并行分支 | 同时执行分析和摘要 |
| `RunnableLambda` | 自定义函数 | 中间数据转换 |
| `.bind(stop=["\n"])` | 绑定参数 | 给 LLM 绑定固定参数 |

## 面试速记

| 问题 | 要点 |
|------|------|
| LCEL 的优势 | 统一接口(invoke/stream/batch)、自动并发、类型安全、可组合 |
| OutputParser 类型 | StrOutputParser, JsonOutputParser, PydanticOutputParser |
| with_structured_output | 比 OutputParser 更简洁，直接返回 Pydantic 对象 |

**下一节**: `02_langchain_tools_agents.ipynb`